# Alexandria End-to-End Provider Walkthrough

This notebook is a manual smoke check for the full configured lifecycle: ingest, retrieve, split-check handling, durable split inspection, and retrieval after maintenance. It uses the configured Postgres database, SQL repositories, unit of work, outbox repository, worker boundary, embedder, summarizer, splitter, search, and ranker settings. Application logging is enabled so slow provider-backed steps are visible while the cells run.

## Prerequisites

Expected setup:
- Docker and Task are available locally.
- The configured Postgres database is reachable through `.env` or the environment.
- Provider credentials are present for the configured embedder, summarizer, and splitter.
- `ALEXANDRIA_SPLITTER__PROVIDER` is set to a provider such as `openai`, and the splitter returns child embeddings compatible with the configured vector dimension.
- The notebook is run from the repository root or from the `sandbox/` directory.

The source keys include a timestamp so the rows are easy to identify in a developer database. The split-check cell handles only the outbox jobs created for this notebook's ingested documents, so unrelated pending jobs are left alone.

## Start Infrastructure

This cell starts the local infrastructure profile through the project Taskfile and waits for Docker Compose health checks before the smoke path runs.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess


repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent
os.chdir(repo)

deploy = subprocess.run(
    ["task", "deploy", "--", "--wait"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if deploy.stdout:
    print(deploy.stdout)
if deploy.stderr:
    print(deploy.stderr)


## Build The Real App

This cell builds the configured `App` directly. The ingest fullness threshold is lowered for this notebook so the newly ingested documents create split-check work immediately.

In [ ]:
from datetime import datetime, timezone
from time import perf_counter
import logging

from sqlalchemy import select

from application.app import App
from application.ports import DocIn
from domain.entity import Document, Job, Node
from domain.values import JobKind, JobStatus
from infrastructure.config import IngestSettings, SplitterProvider, get_settings
from infrastructure.observability.logger import LoggingService
from infrastructure.persistence.db import Db
from infrastructure.repositories.outbox import OutboxRepo
from presentation.worker.app import Worker


get_settings.cache_clear()
base_settings = get_settings()
settings = base_settings.model_copy(
    update={
        "ingest": IngestSettings(max_leaf_docs=1),
        "logging": base_settings.logging.model_copy(
            update={"level": "DEBUG", "stream_handler_enabled": True}
        ),
    }
)
LoggingService.setup(settings.logging)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.INFO)
logging.getLogger("openai").setLevel(logging.WARNING)
log = logging.getLogger("sandbox.06_end_to_end")


async def step(label, awaitable):
    log.info("%s started", label)
    started = perf_counter()
    try:
        result = await awaitable
    except Exception:
        log.exception("%s failed after %.2fs", label, perf_counter() - started)
        raise
    log.info("%s finished in %.2fs", label, perf_counter() - started)
    return result

if settings.splitter.provider == SplitterProvider.NONE:
    raise RuntimeError(
        "Configure ALEXANDRIA_SPLITTER__PROVIDER for this provider-backed end-to-end notebook."
    )
if settings.splitter.api_key is None or not settings.splitter.api_key.get_secret_value().strip():
    raise RuntimeError(
        "Configure ALEXANDRIA_SPLITTER__API_KEY for this provider-backed end-to-end notebook."
    )

db = Db(settings.database)
db.create_all()
app = App(settings)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
marker = f"alexandria end-to-end smoke marker {stamp}"
source_alpha = f"notebook:e2e:alpha:{stamp}"
source_beta = f"notebook:e2e:beta:{stamp}"
source_gamma = f"notebook:e2e:gamma:{stamp}"
source_keys = {source_alpha, source_beta, source_gamma}

print("app", app.name, app.version)
print("embedding", settings.embedding.provider, settings.embedding.model)
print("summarizer", settings.summarizer.provider, settings.summarizer.model)
print("splitter", settings.splitter.provider, settings.splitter.model)
print("ranker", settings.ranker.provider, settings.ranker.model)
print("marker", marker)
print("sources", sorted(source_keys))
log.info(
    "notebook app configured embedding=%s summarizer=%s splitter=%s ranker=%s",
    settings.embedding.provider,
    settings.summarizer.provider,
    settings.splitter.provider,
    settings.ranker.provider,
)


## Prepare Documents

This cell creates three timestamped documents so the provider-backed split lifecycle has enough local content to inspect after maintenance.

In [ ]:
docs = [
    DocIn(
        name=f"Notebook E2E Alpha {stamp}",
        body=f"Alpha lifecycle document for {marker}. It covers setup, routing, retrieval, and maintenance.",
        source_key=source_alpha,
    ),
    DocIn(
        name=f"Notebook E2E Beta {stamp}",
        body=f"Beta operations document for {marker}. It covers outbox jobs, worker handling, and retries.",
        source_key=source_beta,
    ),
    DocIn(
        name=f"Notebook E2E Gamma {stamp}",
        body=f"Gamma split document for {marker}. It covers child leaves, document movement, and post-split retrieval.",
        source_key=source_gamma,
    ),
]
print("documents", [(doc.name, doc.source_key) for doc in docs])


## Seed And Ingest

This cell exercises the real ingest path through `App.ingest(...)`: summarize, embed, route, persist, update counts, and append split-check work. It prints progress after each provider-backed ingest call so a slow provider is visible.

In [ ]:
root = await step("seed", app.seed())
ids = []
for doc in docs:
    id = await step(f"ingest {doc.source_key}", app.ingest(doc))
    ids.append(id)
    print("ingested", doc.source_key, id, flush=True)

print("root", root.id, root.kind, root.status)
print("document ids", [str(id) for id in ids])


## Retrieve Before Maintenance

This cell runs retrieval separately from ingest so the query embedding and optional ranker call have their own visible logging.

In [ ]:
app.session.expire_all()
before_hits = await step("retrieve before maintenance", app.retrieve(marker, limit=10))
before_sources = [hit.doc.source_key for hit in before_hits]

print("before hit sources", before_sources)

assert any(source in source_keys for source in before_sources), (
    "retrieve before maintenance did not return a notebook document"
)


## Inspect State Before Split Handling

Expected notes:
- The three timestamped documents should exist with summaries and embeddings from the configured providers.
- Each document should be attached to an active leaf before split handling.
- At least one `split.check` outbox job should exist for the leaves touched by these documents because the notebook lowered `max_leaf_docs` to `1`.

In [ ]:
with db.session() as session:
    persisted = session.scalars(
        select(Document)
        .where(Document.source_key.in_(source_keys))
        .order_by(Document.source_key.asc())
    ).all()
    assert len(persisted) == len(source_keys), "not all notebook documents were persisted"

    leaf_ids = {doc.leaf_id for doc in persisted}
    leaves = session.scalars(
        select(Node).where(Node.id.in_(leaf_ids)).order_by(Node.created_at.asc())
    ).all()
    jobs = session.scalars(
        select(Job)
        .where(Job.kind == JobKind.SPLIT_CHECK.value, Job.key.in_(leaf_ids))
        .order_by(Job.created_at.asc())
    ).all()
    assert jobs, "no split.check jobs were queued for the notebook leaves"

    print("documents")
    for document in persisted:
        embedding = list(document.embedding)
        print(
            " ",
            document.source_key,
            str(document.id),
            "leaf",
            document.leaf_id,
            "summary",
            document.summary[:120],
            "embedding_dims",
            len(embedding),
            "embedding_first_5",
            [round(value, 6) for value in embedding[:5]],
        )

    print("leaves", [(str(leaf.id), leaf.kind, leaf.status, leaf.doc_count) for leaf in leaves])
    print("jobs", [(str(job.id), str(job.key), job.status, job.payload) for job in jobs])

    assert all(document.summary for document in persisted)
    assert all(len(list(document.embedding)) > 0 for document in persisted)
    assert all(leaf.kind == "leaf" and leaf.status == "active" for leaf in leaves)

job_ids = [job.id for job in jobs]
job_leaf_ids = {job.key for job in jobs}


## Handle Notebook Split-Check Jobs

This cell handles only the split-check jobs queued for this notebook's leaves. It marks each known job as running, calls the real worker handler, and then marks the job done through the real outbox repository. Unrelated pending jobs in the developer database are not claimed.

In [ ]:
worker = Worker(app=app, kind=JobKind.SPLIT_CHECK)
processed_job_ids = []

for job_id in job_ids:
    with db.session() as session:
        claimed = session.get(Job, job_id)
        assert claimed is not None
        if claimed.status != JobStatus.PENDING.value:
            print("skip job", job_id, claimed.status)
            continue

        claimed.status = JobStatus.RUNNING.value
        claimed.locked_at = datetime.now(timezone.utc)
        claimed.attempts += 1
        session.flush()
        worker_job = Job(
            id=claimed.id,
            kind=claimed.kind,
            payload=dict(claimed.payload),
            key=claimed.key,
        )
        session.commit()

    try:
        await step(f"worker handle split.check {job_id}", worker.handle(worker_job))
    except Exception as error:
        with db.session() as session:
            outbox = OutboxRepo(session, settings.queue)
            await outbox.mark(job_id, JobStatus.FAILED, str(error))
            session.commit()
        raise
    else:
        with db.session() as session:
            outbox = OutboxRepo(session, settings.queue)
            await outbox.mark(job_id, JobStatus.DONE)
            session.commit()
        processed_job_ids.append(job_id)

app.session.expire_all()

print("processed jobs", [str(id) for id in processed_job_ids])
assert processed_job_ids, "no notebook split-check jobs were processed"


## Inspect State After Split Handling

Expected notes:
- Each processed source leaf should now be an active branch.
- Child leaves should exist below the processed source leaves.
- The timestamped documents should now be attached to child leaves.
- The notebook jobs should be marked done.

In [ ]:
with db.session() as session:
    processed_jobs = session.scalars(
        select(Job).where(Job.id.in_(processed_job_ids)).order_by(Job.created_at.asc())
    ).all()
    processed_leaf_ids = {job.key for job in processed_jobs}
    sources = session.scalars(
        select(Node).where(Node.id.in_(processed_leaf_ids)).order_by(Node.created_at.asc())
    ).all()
    children = session.scalars(
        select(Node).where(Node.parent_id.in_(processed_leaf_ids)).order_by(Node.created_at.asc())
    ).all()
    child_ids = {child.id for child in children}
    moved_docs = session.scalars(
        select(Document)
        .where(Document.source_key.in_(source_keys))
        .order_by(Document.source_key.asc())
    ).all()

    print("sources", [(str(node.id), node.kind, node.status, node.doc_count, node.version) for node in sources])
    print("children", [(str(node.id), str(node.parent_id), node.name, node.kind, node.status, node.doc_count) for node in children])
    print("moved docs", [(doc.source_key, str(doc.leaf_id)) for doc in moved_docs])
    print("jobs", [(str(job.id), job.status, job.done_at is not None, job.last_error) for job in processed_jobs])

    assert sources
    assert children
    assert all(node.kind == "branch" and node.status == "active" for node in sources)
    assert all(node.kind == "leaf" and node.status == "active" for node in children)
    assert all(doc.leaf_id in child_ids for doc in moved_docs)
    assert all(job.status == JobStatus.DONE.value for job in processed_jobs)
    assert all(job.done_at is not None for job in processed_jobs)
    assert all(job.last_error is None for job in processed_jobs)


## Retrieve After Maintenance

This cell proves retrieval still finds the notebook documents after the split-check lifecycle moved them to child leaves.

In [ ]:
after_hits = await step("retrieve after maintenance", app.retrieve(marker, limit=10))
after_sources = [hit.doc.source_key for hit in after_hits]

print("after hit sources", after_sources)
for hit in after_hits:
    if hit.doc.source_key in source_keys:
        print(
            "hit",
            {
                "source_key": hit.doc.source_key,
                "name": hit.doc.name,
                "score": round(hit.score, 6),
                "distance": round(hit.distance or 0.0, 6),
                "bm25": hit.bm25,
            },
        )

assert any(source in source_keys for source in after_sources), (
    "retrieve after maintenance did not return a notebook document"
)


### Expected Smoke Output

- Ingest should print three document ids and retrieval before maintenance should include at least one timestamped source key.
- Before split handling, the timestamped documents should have provider summaries, provider embeddings, active leaves, and queued split-check jobs.
- Split handling should process only the notebook job ids.
- After split handling, processed source leaves should be active branches with active child leaves, moved documents, and done outbox jobs.
- Retrieval after maintenance should still include at least one timestamped source key.

### Matching Pytest Smoke Command

Run this from the repository root:

```bash
uv run pytest tests/integration/test_end_to_end_flow.py -q
```

The pytest smoke remains deterministic for automated evaluation. This notebook is the provider-backed manual path.

## Tear Down Infrastructure

Run this final cell when the smoke check is complete. It closes local application resources and stops the infrastructure profile through the project Taskfile without deleting Docker volumes.

In [ ]:
from pathlib import Path
import subprocess


app.close()

repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent

teardown = subprocess.run(
    ["task", "teardown"],
    cwd=repo,
    text=True,
    capture_output=True,
    check=True,
)

if teardown.stdout:
    print(teardown.stdout)
if teardown.stderr:
    print(teardown.stderr)
